Tenglish Next-Word Autocomplete (Gboard-style)

Same architecture as the Tanglish notebook: fine-tuned **DistilGPT2** + **n-gram** model, blended with a weight tuned on validation data. It ranks whole words, never samples free text.

**Before pressing Save & Run All**
1. Add your private dataset (the `tenglish*.csv` file(s)) via *Add Input*.
2. Session options: **Accelerator = GPU**, **Internet = On** (needed once to download distilgpt2).
3. The notebook has no interactive cells, so it runs top to bottom on its own.

**Outputs** (Output tab after the run): `tenglish-autocomplete.zip`, the model folder to download and use locally.

In [ ]:
# ============================================================
#  CONFIG  (usually nothing to change)
# ============================================================
MODEL_NAME = "distilgpt2"

# Leave empty to auto-detect every tenglish*.csv under /kaggle/input.
# Or set explicitly, e.g. CSV_PATHS = ["/kaggle/input/tenglish-dataset/tenglish_train.csv"]
CSV_PATHS = []

SEED       = 42
VAL_FRAC   = 0.10          # held out to tune the LM/n-gram mix and pick the epoch
TEST_FRAC  = 0.10          # held out for the final honest accuracy numbers

# Fine-tuning (same values as the Tanglish notebook, so the two models are comparable)
MAX_EPOCHS    = 30
BATCH_SIZE    = 16
LEARNING_RATE = 1e-4
WEIGHT_DECAY  = 0.01
WARMUP_FRAC   = 0.1
PATIENCE      = 4

NGRAM_DISCOUNT = 0.75
TOP_K = 3

# Optional spelling merge, applied to every word in train AND at typing time.
# Fill this AFTER your first run, from the "Most common words" printout.
# Only merge variants of the same word, e.g.:
# NORMALIZE = {"chestunna": "chesthunna", "vastunna": "vasthunna"}
NORMALIZE = {}

OUT_DIR = "/kaggle/working/tenglish-autocomplete"   # falls back to ./tenglish-autocomplete off Kaggle

In [ ]:
import os, re, json, glob, math, random, copy, zipfile
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if not os.path.isdir(os.path.dirname(OUT_DIR)):
    OUT_DIR = "./tenglish-autocomplete"
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no GPU (works, but slower)")

 1. Load and clean the data
Lower-cased, punctuation stripped, exactly like the Tanglish notebook. Rows whose category starts with `synthetic` are used for **training only**, so the validation/test numbers are measured on real chat.

In [ ]:
def tokenize_words(text):
    """'Ela unnav ra?' -> ['ela', 'unnav', 'ra']"""
    text = re.sub(r"(?<=[a-z0-9])\.(?=[a-z0-9])", "", str(text).lower())
    words = re.findall(r"[a-z0-9']+(?:-[a-z0-9']+)*", text)
    return [NORMALIZE.get(w, w) for w in words]

def find_csvs():
    if CSV_PATHS:
        return CSV_PATHS
    pats = ["/kaggle/input/**/tenglish*.csv", "/content/**/tenglish*.csv",
            "./**/tenglish*.csv", "/mnt/user-data/uploads/tenglish*.csv"]
    found = []
    for p in pats:
        found += glob.glob(p, recursive=True)
    return sorted(set(found))

paths = find_csvs()
if not paths:
    raise FileNotFoundError("No tenglish*.csv found. Attach your dataset via Add Input, or set CSV_PATHS in the config cell.")
print("CSV files:", paths)

frames = []
for p in paths:
    d = pd.read_csv(p)
    col = "sentence" if "sentence" in d.columns else ("text" if "text" in d.columns else None)
    if col is None:
        raise ValueError(f"{p}: needs a 'sentence' (or 'text') column, found {list(d.columns)}")
    d = d.rename(columns={col: "sentence"})
    if "category" not in d.columns:
        d["category"] = "synthetic" if "synth" in os.path.basename(p).lower() else "all"
    frames.append(d[["category", "sentence"]])
df = pd.concat(frames, ignore_index=True).dropna(subset=["sentence"])
df["category"] = df["category"].astype(str).str.strip().str.lower()
df["words"] = df["sentence"].map(tokenize_words)
df = df[df["words"].map(len) > 0]
df["key"] = df["words"].map(" ".join)
before = len(df)
df = df.drop_duplicates("key").reset_index(drop=True)
print(f"Rows: {before} -> {len(df)} after removing duplicates")
print("Sentences per category:", dict(Counter(df["category"])))
if len(df) < 500:
    print("WARNING: fewer than 500 sentences. The model will run, but expect weak suggestions.")

all_counts = Counter(w for ws in df["words"] for w in ws)
print(f"Unique words: {len(all_counts)} | seen only once: {sum(1 for c in all_counts.values() if c == 1)}")
print("Sentence length (words):", dict(sorted(Counter(df['words'].map(len)).items())))
print("Most common words:", all_counts.most_common(40))

 2. Split by sentence (stratified by category / chat)
Each chat is split separately, so validation and test contain a bit of every chat. `synthetic*` categories go entirely to training.

In [ ]:
def stratified_split(frame, val_frac, test_frac, seed):
    rng = random.Random(seed)
    tr, va, te = [], [], []
    for cat, g in frame.groupby("category"):
        idx = list(g.index); rng.shuffle(idx)
        if cat.startswith("synthetic"):
            tr += idx
            continue
        n = len(idx)
        nt = max(1, round(n * test_frac)) if n >= 5 else 0
        nv = max(1, round(n * val_frac)) if n >= 5 else 0
        te += idx[:nt]; va += idx[nt:nt + nv]; tr += idx[nt + nv:]
    return tr, va, te

tr_idx, va_idx, te_idx = stratified_split(df, VAL_FRAC, TEST_FRAC, SEED)
if not va_idx or not te_idx:
    raise ValueError("No real (non-synthetic) category has 5+ sentences, so nothing was held out for validation/testing.")
train_sents = df.loc[tr_idx, "words"].tolist()
val_sents   = df.loc[va_idx, "words"].tolist()
test_sents  = df.loc[te_idx, "words"].tolist()
all_sents   = df["words"].tolist()
print(f"train={len(train_sents)}  val={len(val_sents)}  test={len(test_sents)}")

train_vocab = {w for s in train_sents for w in s}
for name, ss in [("val", val_sents), ("test", test_sents)]:
    toks = [w for s in ss for w in s]
    oov = sum(1 for w in toks if w not in train_vocab) / max(1, len(toks))
    print(f"{name}: {oov:.0%} of words never appear in train (nobody can predict these; they cap the accuracy)")

 3. N-gram model (interpolated absolute discounting, trigram)

In [ ]:
BOS, EOS_W = "<s>", "</s>"

class NGram:
    def __init__(self, sents, vocab, discount=0.75):
        self.vocab = vocab
        self.w2i = {w: i for i, w in enumerate(vocab)}
        self.w2i[EOS_W] = len(vocab)
        self.V = len(vocab)
        self.d = discount
        self.c1 = defaultdict(Counter)
        self.c2 = defaultdict(Counter)
        left = defaultdict(set)
        for s in sents:
            seq = [BOS, BOS] + s + [EOS_W]
            for i in range(2, len(seq)):
                self.c1[(seq[i - 1],)][seq[i]] += 1
                self.c2[(seq[i - 2], seq[i - 1])][seq[i]] += 1
                left[seq[i]].add(seq[i - 1])
        uni = np.full(self.V + 1, 0.1)
        for w, i in self.w2i.items():
            uni[i] += len(left.get(w, ()))
        self.uni = uni / uni.sum()

    def _interp(self, lower, counts):
        if not counts:
            return lower
        total = sum(counts.values())
        out = lower * (self.d * len(counts) / total)
        for w, n in counts.items():
            j = self.w2i.get(w)
            if j is not None:
                out[j] += max(n - self.d, 0.0) / total
        return out

    def probs(self, ctx):
        seq = [BOS, BOS] + list(ctx)
        p = self.uni.copy()
        p = self._interp(p, self.c1.get((seq[-1],)))
        p = self._interp(p, self.c2.get((seq[-2], seq[-1])))
        p = p[:self.V]
        return p / p.sum()

 4. Fine-tune DistilGPT2 (plain PyTorch loop)

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.pad_token = tok.eos_token
EOS = tok.eos_token_id

def encode_sentence(words):
    return [EOS] + tok.encode(" ".join(words)) + [EOS]

def make_batch(seqs):
    n, L = len(seqs), max(len(s) for s in seqs)
    ids = torch.full((n, L), EOS, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    lab = torch.full((n, L), -100, dtype=torch.long)
    for r, s in enumerate(seqs):
        t = torch.tensor(s)
        ids[r, :len(s)] = t
        att[r, :len(s)] = 1
        lab[r, 1:len(s)] = t[1:]
    return ids.to(DEVICE), att.to(DEVICE), lab.to(DEVICE)

def lm_loss_sum(model, seqs):
    ids, att, lab = make_batch(seqs)
    logits = model(input_ids=ids, attention_mask=att).logits
    loss = F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)).float(),
                           lab[:, 1:].reshape(-1), ignore_index=-100, reduction="sum")
    return loss, int((lab[:, 1:] != -100).sum())

@torch.no_grad()
def eval_loss(model, seqs, bs=64):
    model.eval()
    tot, n = 0.0, 0
    for i in range(0, len(seqs), bs):
        l, c = lm_loss_sum(model, seqs[i:i + bs])
        tot += l.item(); n += c
    return tot / n

def train_lm(train_words, val_words=None, epochs=MAX_EPOCHS, patience=PATIENCE):
    """With val_words: early-stops and returns the best epoch. Without: trains exactly `epochs`."""
    torch.manual_seed(SEED)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
    tr = [encode_sentence(w) for w in train_words]
    va = [encode_sentence(w) for w in val_words] if val_words else None
    steps_per_epoch = math.ceil(len(tr) / BATCH_SIZE)
    total = steps_per_epoch * epochs
    opt = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP_FRAC * total), total)
    best_loss, best_state, best_epoch, bad = float("inf"), None, epochs, 0
    for ep in range(1, epochs + 1):
        model.train()
        order = list(range(len(tr))); random.shuffle(order)
        run, cnt = 0.0, 0
        for i in range(0, len(order), BATCH_SIZE):
            loss_sum, n = lm_loss_sum(model, [tr[j] for j in order[i:i + BATCH_SIZE]])
            (loss_sum / n).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
            run += loss_sum.item(); cnt += n
        msg = f"epoch {ep:2d}  train_loss={run / cnt:.4f}"
        if va is not None:
            vl = eval_loss(model, va)
            msg += f"  val_loss={vl:.4f}  val_ppl={math.exp(vl):.1f}"
            if vl < best_loss - 1e-4:
                best_loss, best_epoch, bad = vl, ep, 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                msg += "  *best*"
            else:
                bad += 1
        print(msg)
        if va is not None and bad >= patience:
            print(f"Early stop (no improvement for {patience} epochs).")
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    return model, best_epoch

print(f"Fine-tuning on {len(train_sents)} sentences ...")
lm_model, best_epoch = train_lm(train_sents, val_sents)
print("Best epoch:", best_epoch)

 5. Score whole words with the language model

In [ ]:
class LMScorer:
    def __init__(self, model, vocab, chunk=128):
        self.model, self.vocab, self.chunk = model.eval(), vocab, chunk
        self.first = [tok.encode(w) for w in vocab]
        self.later = [tok.encode(" " + w) for w in vocab]
        n_tok = model.get_output_embeddings().weight.shape[0]
        boundary = []
        for i in range(min(n_tok, len(tok))):
            s = tok.decode([i])
            if i == EOS or (s and (s[0].isspace() or not (s[0].isalnum() or s[0] == "'"))):
                boundary.append(i)
        self.boundary = torch.tensor(boundary, device=DEVICE)
        self.cache = {}

    @torch.no_grad()
    def log_probs(self, ctx):
        key = tuple(ctx)
        if key in self.cache:
            return self.cache[key]
        prefix = [EOS] + (tok.encode(" ".join(ctx)) if ctx else [])
        P = len(prefix)
        cands = self.later if ctx else self.first
        order = sorted(range(len(cands)), key=lambda i: len(cands[i]))
        scores = np.zeros(len(cands))
        for a in range(0, len(order), self.chunk):
            idx = order[a:a + self.chunk]
            n, L = len(idx), max(len(cands[i]) for i in idx)
            ids = torch.full((n, P + L), EOS, dtype=torch.long)
            att = torch.zeros((n, P + L), dtype=torch.long)
            tgt = torch.zeros((n, L), dtype=torch.long)
            msk = torch.zeros((n, L))
            last = torch.zeros(n, dtype=torch.long)
            for r, i in enumerate(idx):
                seq = prefix + cands[i]
                ids[r, :len(seq)] = torch.tensor(seq)
                att[r, :len(seq)] = 1
                tgt[r, :len(cands[i])] = torch.tensor(cands[i])
                msk[r, :len(cands[i])] = 1
                last[r] = len(seq) - P
            ids, att, tgt, msk, last = (t.to(DEVICE) for t in (ids, att, tgt, msk, last))
            logits = self.model(input_ids=ids, attention_mask=att).logits[:, P - 1:, :]
            lp = F.log_softmax(logits.float(), dim=-1)
            tok_lp = lp[:, :L, :].gather(2, tgt.unsqueeze(-1)).squeeze(-1)
            word_lp = (tok_lp * msk).sum(1)
            end_lp = torch.logsumexp(lp[torch.arange(n, device=DEVICE), last][:, self.boundary], dim=-1)
            scores[idx] = (word_lp + end_lp).cpu().numpy()
        scores = scores - np.logaddexp.reduce(scores)
        self.cache[key] = scores
        return scores

    def probs(self, ctx):
        return np.exp(self.log_probs(ctx))

 6. The suggester (keyboard logic)

In [ ]:
class Suggester:
    def __init__(self, vocab, ngram, lm=None, alpha=0.5):
        self.vocab, self.ng, self.lm, self.alpha = vocab, ngram, lm, alpha
        self.w2i = {w: i for i, w in enumerate(vocab)}
        self._starts = {}

    def blend(self, p_lm, p_ng, alpha=None):
        a = self.alpha if alpha is None else alpha
        return a * p_lm + (1 - a) * p_ng

    def prefix_mask(self, prefix):
        if prefix not in self._starts:
            self._starts[prefix] = np.array([w.startswith(prefix) for w in self.vocab])
        return self._starts[prefix]

    def probs(self, ctx):
        p_ng = self.ng.probs(ctx)
        if self.lm is None:
            return p_ng
        return self.blend(self.lm.probs(ctx), p_ng)

    def suggest(self, text, k=TOP_K):
        """'nuvvu ' -> next-word suggestions.  'nuvvu vas' -> completions of 'vas'."""
        words = tokenize_words(text)
        typing_word = bool(text) and not text[-1].isspace() and bool(words)
        prefix = words[-1] if typing_word else ""
        ctx = words[:-1] if typing_word else words
        p = self.probs(ctx)
        if prefix:
            mask = self.prefix_mask(prefix)
            if not mask.any():
                return [prefix]
            p = np.where(mask, p, -1.0)
        top = np.argsort(-p)[:k]
        return [self.vocab[i] for i in top if p[i] >= 0]

 7. Evaluate on held-out real sentences and tune the LM / n-gram mix

In [ ]:
def build_positions(sents, sug):
    rows = []
    for s in sents:
        for i, w in enumerate(s):
            ctx = s[:i]
            rows.append(dict(ctx=ctx, target=w,
                             p_ng=sug.ng.probs(ctx),
                             p_lm=sug.lm.probs(ctx) if sug.lm else None))
    return rows

def score_rows(rows, sug, combine, prefix_len=0, ks=(1, 3, 5)):
    hits = {k: 0 for k in ks}; rr = 0.0; n = 0; nll = 0.0; n_in = 0
    for r in rows:
        t = r["target"]
        if prefix_len and len(t) <= prefix_len:
            continue
        n += 1
        j = sug.w2i.get(t)
        if j is None:
            continue
        p = combine(r)
        if prefix_len:
            p = np.where(sug.prefix_mask(t[:prefix_len]), p, -1.0)
        else:
            nll -= math.log(max(p[j], 1e-12)); n_in += 1
        rank = 1 + int((p > p[j]).sum())
        rr += 1.0 / rank
        for k in ks:
            hits[k] += rank <= k
    out = {f"top{k}": hits[k] / max(n, 1) for k in ks}
    out["MRR"] = rr / max(n, 1)
    out["NLL"] = nll / n_in if n_in else float("nan")
    out["n"] = n
    return out

def tune_alpha(rows, sug):
    best = (None, float("inf"))
    for a in np.round(np.arange(0, 1.01, 0.1), 2):
        m = score_rows(rows, sug, lambda r: a * r["p_lm"] + (1 - a) * r["p_ng"])
        if m["NLL"] < best[1]:
            best = (float(a), m["NLL"])
    return best[0]

vocab_train = sorted(train_vocab)
ngram = NGram(train_sents, vocab_train, NGRAM_DISCOUNT)
lm_scorer = LMScorer(lm_model, vocab_train)
sug = Suggester(vocab_train, ngram, lm_scorer)

print("Scoring validation positions ...")
val_rows = build_positions(val_sents, sug)
best_alpha = tune_alpha(val_rows, sug)
sug.alpha = best_alpha
print(f"Best LM weight on validation (lowest NLL): alpha = {best_alpha}  (0 = n-gram only, 1 = DistilGPT2 only)")

print("Scoring test positions ...")
test_rows = build_positions(test_sents, sug)
uni_p = ngram.uni[:ngram.V] / ngram.uni[:ngram.V].sum()
models = {
    "Most-frequent words (baseline)": lambda r: uni_p,
    "N-gram only":                    lambda r: r["p_ng"],
    "DistilGPT2 only":                lambda r: r["p_lm"],
    f"HYBRID (alpha={best_alpha})":    lambda r: best_alpha * r["p_lm"] + (1 - best_alpha) * r["p_ng"],
}
lines = []
for name, fn in models.items():
    for pl, label in [(0, "next word (nothing typed)"), (1, "1st letter typed"), (2, "2 letters typed")]:
        m = score_rows(test_rows, sug, fn, prefix_len=pl)
        lines.append({"model": name, "situation": label, "top-1": m["top1"], "top-3": m["top3"],
                      "top-5": m["top5"], "MRR": m["MRR"], "positions": m["n"]})
report = pd.DataFrame(lines)
pd.set_option("display.width", 200)
print("\nTEST-SET RESULTS (real sentences the models never saw)")
print(report.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
report.to_csv(os.path.join(os.path.dirname(OUT_DIR.rstrip("/")) or ".", "tenglish_test_results.csv"), index=False)

 8. Refit on ALL your data and save
The final model is retrained on 100% of the data (train + validation + test + synthetic) for the best epoch count found above.

**Privacy:** `tenglish_meta.json` stores the training sentences (the n-gram model is rebuilt from them at load time). Keep the output zip private.

In [ ]:
final_vocab = sorted({w for s in all_sents for w in s})
print(f"Refitting DistilGPT2 on all {len(all_sents)} sentences for {best_epoch} epochs ...")
final_lm, _ = train_lm(all_sents, None, epochs=best_epoch)

final_sug = Suggester(final_vocab, NGram(all_sents, final_vocab, NGRAM_DISCOUNT),
                      LMScorer(final_lm, final_vocab), alpha=best_alpha)

os.makedirs(OUT_DIR, exist_ok=True)
final_lm.save_pretrained(OUT_DIR)
tok.save_pretrained(OUT_DIR)
with open(os.path.join(OUT_DIR, "tenglish_meta.json"), "w") as f:
    json.dump({"sentences": all_sents, "alpha": best_alpha, "discount": NGRAM_DISCOUNT,
               "base_model": MODEL_NAME, "best_epoch": best_epoch, "normalize": NORMALIZE}, f, ensure_ascii=False)

zip_path = OUT_DIR.rstrip("/") + ".zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT_DIR):
        for fn in files:
            full = os.path.join(root, fn)
            z.write(full, os.path.relpath(full, os.path.dirname(OUT_DIR)))
print("Saved:", OUT_DIR, "\nZip  :", zip_path, f"({os.path.getsize(zip_path) / 1e6:.0f} MB)")

 9. Try it
A trailing space means "suggest the next word". No trailing space means "complete the word I'm typing". Edit the list to use words that actually appear in your data.

In [ ]:
tests = ["", "em ", "nuvvu ", "repu ", "ela ", "nenu ", "chala ", "ela unnav ", "class ", "vasthava ", "nen", "ra"]
for t in tests:
    print(f"{t!r:<18} -> {final_sug.suggest(t)}")

 10. Reload check + how to use the saved model later
This proves the saved folder loads and gives the same suggestions, which is exactly what the local app / middleware will do.

In [ ]:
def load_suggester(model_dir, device=DEVICE):
    global tok, EOS, DEVICE, NORMALIZE
    DEVICE = device
    tok = AutoTokenizer.from_pretrained(model_dir); tok.pad_token = tok.eos_token; EOS = tok.eos_token_id
    lm = AutoModelForCausalLM.from_pretrained(model_dir).to(DEVICE).eval()
    meta = json.load(open(os.path.join(model_dir, "tenglish_meta.json")))
    NORMALIZE = meta.get("normalize", {})
    vocab = sorted({w for s in meta["sentences"] for w in s})
    return Suggester(vocab, NGram(meta["sentences"], vocab, meta["discount"]),
                     LMScorer(lm, vocab), alpha=meta["alpha"])

reloaded = load_suggester(OUT_DIR)
for t in ["nuvvu ", "ela unnav ", "em "]:
    print(f"{t!r:<18} original={final_sug.suggest(t)}  reloaded={reloaded.suggest(t)}")